In [1]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [2]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble

In [3]:

from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [4]:
# Plotting utils
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

# Load all data from train/valid/test

In [5]:
# Path to your data folder
data_path = "data_product_reviews/"


### Load training

In [6]:

# Load features and labels
X_train = pd.read_csv(data_path + "x_train.csv")
y_train = pd.read_csv(data_path + "y_train.csv").squeeze()


### Load validation set

In [7]:
X_valid = pd.read_csv(data_path + "x_valid.csv")
y_valid = pd.read_csv(data_path + "y_valid.csv").squeeze()

### Load test set 

In [8]:
X_test = pd.read_csv(data_path + "x_test.csv")
y_test = pd.read_csv(data_path + "y_test.csv").squeeze()

print("Data loaded successfully!")
print(f"Training: {X_train.shape}, Validation: {X_valid.shape}, Test: {X_test.shape}")



Data loaded successfully!
Training: (6346, 7729), Validation: (792, 7729), Test: (793, 7729)


### Load vocabulary as a list of strings

In [9]:
### Load vocabulary as a list of strings (if provided)
# Some datasets include vocab.txt or vocab.csv; if not, skip this step.
vocab_file = data_path + "vocab.csv"  # or "vocab.txt" if that's your file
if os.path.exists(vocab_file):
    vocab_list = pd.read_csv(vocab_file, header=None).squeeze().tolist()
    print(f"Loaded vocabulary with {len(vocab_list)} words.")
else:
    vocab_list = None
    print("No vocabulary file found")

No vocabulary file found


In [10]:
### Load vocabulary as a list of strings (derived from training features)

vocab_list = list(X_train.columns)
print(f"Derived vocabulary from training data with {len(vocab_list)} features.")
print("Example tokens:", vocab_list[:20])


Derived vocabulary from training data with 7729 features.
Example tokens: ['good', 'great', 'time', 'book', "don't", 'work', 'i_have', 'read', 'make', 'if_you', 'thing', 'buy', 'i_was', 'and_i', 'this_book', 'product', 'movie', 'it_was', 'love', 'review']


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [12]:
from sklearn.model_selection import PredefinedSplit

In [13]:

# # # Combine data
X_all = pd.concat([X_train, X_valid], axis=0).reset_index(drop=True)
y_all = pd.concat([y_train, y_valid], axis=0).reset_index(drop=True)


In [14]:
# Create predefined split: -1 for training, 0 for validation
test_fold = np.concatenate([
    -1 * np.ones(X_train.shape[0], dtype=int),  # training samples
     0 * np.ones(X_valid.shape[0], dtype=int)   # validation samples
])
my_splitter = PredefinedSplit(test_fold)


In [15]:

print("Combined data ready for GridSearchCV")
print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)


Combined data ready for GridSearchCV
X_all shape: (7138, 7729)
y_all shape: (7138,)


# Part 2: Decision Trees

## 2.1: Train a simple tree with depth 3

In [16]:
from sklearn.tree import DecisionTreeClassifier

In [17]:

# Define and train the base Decision Tree
base_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)

### **Fit the tree** 

**TODO Train on the training set** in the next coding cell

In [18]:
# Train on the training set
base_tree.fit(X_train, y_train)

print("Trained Decision Tree (max_depth=3)")
print(f"Training accuracy: {base_tree.score(X_train, y_train):.3f}")
print(f"Validation accuracy: {base_tree.score(X_valid, y_valid):.3f}")



Trained Decision Tree (max_depth=3)
Training accuracy: 0.646
Validation accuracy: 0.649


### **Figure 1: Print Tree** 

Use a helper function from the starter code

In [19]:
### Figure 1: Print Tree

pretty_print_sklearn_tree(base_tree, feature_names=vocab_list)


The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (1 total training examples)
    N Decision: X['movie'] <= 0.50?
      Y Leaf: p(y

#### Is there any internal node that has two child leaf nodes corresponding to the same sentiment class? Why would having two children predict the same class make sense?

- Yes — there is at least one internal node whose two child leaves predict the same sentiment class.


- Under the left branch of the tree: the internal test Decision: X['disappoint'] <= 0.50? (the one at depth 2 under X['great'] <= 0.50? → X['excel'] <= 0.50? → Y) has two child leaves with
p(y=1 | this leaf) = 0.430 and p(y=1 | this leaf) = 0.114.
Both probabilities are < 0.5, so both leaves would be classified as negative (y=0).

- Why can this happen?
A decision tree chooses splits to reduce impurity (like Gini), not directly to change the predicted class. Sometimes, a split divides the data into two groups that both have the same majority class (e.g., both negative), but with different class probabilities (e.g., one leaf has 43% positive, the other 11% positive). This split still reduces impurity because it separates examples with different levels of certainty, even if both are classified the same way.



#### Why a split can produce two leaves that predict the same class

- Impurity reduction, not just final class: The tree chooses splits to reduce impurity (Gini) in the children, not directly to change the predicted class. A split can reduce impurity even if both resulting groups still have the same majority class — e.g., one child might be more strongly negative than the other (0.114 vs 0.430), which lowers overall impurity.

- Different confidence / probability estimates: Although both leaves predict negative, their class probabilities differ. That difference can be useful (e.g., gives a confidence score, or matters if we later combine trees or calibrate probabilities).

- Small-sample / interaction effects: With min_samples_leaf=1 and a tiny or noisy training set, the algorithm can isolate single examples (overfitting), producing leaves that happen to agree on class. Also a split that produces same-class leaves may still enable further splits (in a deeper tree) that separate them later based on other features or interactions.

- Practical consequence: Such splits are common with deep or unconstrained trees and small data; they’re often harmless but can indicate overfitting. Pruning or increasing min_samples_leaf / reducing max_depth will reduce these same-class sibling leaves.

- Short summary: yes — the tree contains sibling leaves that predict the same class (negative). This can still make sense because the split reduced impurity and produced different confidence levels or created groups that later splits (or ensembles) can treat differently


## 2.2 : Find best Decision Tree with grid search

#### Hyperparameter Tuning

In [20]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [2, 8, 32, 128],
    "min_samples_leaf": [1, 3, 9],
    "random_state": [101]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(criterion='gini'),
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    n_jobs=-1
)

grid_search.fit(X_all, y_all)

print("Grid search complete.")
print(f"Best balanced accuracy: {grid_search.best_score_:.4f}")
print("Best hyperparameters:", grid_search.best_params_)


Grid search complete.
Best balanced accuracy: 0.7324
Best hyperparameters: {'max_depth': 32, 'min_samples_leaf': 3, 'random_state': 101}


In [21]:

results_df = pd.DataFrame(grid_search.cv_results_)
display(results_df[['param_max_depth', 'param_min_samples_leaf', 'mean_train_score', 'mean_test_score']])

print("Best hyperparameters:", grid_search.best_params_)


,param_max_depth,param_min_samples_leaf,mean_train_score,mean_test_score
0,2,1,0.640429,0.633931
1,2,3,0.640429,0.633931
2,2,9,0.640429,0.633931
3,8,1,0.726505,0.698887
4,8,3,0.721935,0.697599
5,8,9,0.712791,0.696208
6,32,1,0.918197,0.727378
7,32,3,0.877218,0.732380
8,32,9,0.814975,0.713458
9,128,1,0.996216,0.730313


Best hyperparameters: {'max_depth': 32, 'min_samples_leaf': 3, 'random_state': 101}


### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [ ]:
# ### Build the best decision tree

best_dt_params = grid_search.best_params_
# Option A: instantiate a fresh DecisionTreeClassifier with those params
best_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=best_dt_params['max_depth'],
    min_samples_leaf=best_dt_params['min_samples_leaf'],
    random_state=best_dt_params.get('random_state', 101)
)
best_tree.fit(X_train, y_train)

print("Best Decision Tree trained (separate object).")
print(f"Training accuracy: {best_tree.score(X_train, y_train):.3f}")
print(f"Validation accuracy: {best_tree.score(X_valid, y_valid):.3f}")


Best Decision Tree trained (separate object).
Training accuracy: 0.877
Validation accuracy: 0.734


### Interpret the best decision tree

In [23]:
pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)

The binary tree structure has 817 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    0 are leaves
- depth   4 has   16 nodes, of which    6 are leaves
- depth   5 has   20 nodes, of which    5 are leaves
- depth   6 has   30 nodes, of which   14 are leaves
- depth   7 has   32 nodes, of which   18 are leaves
- depth   8 has   28 nodes, of which   12 are leaves
- depth   9 has   32 nodes, of which   11 are leaves
- depth  10 has   42 nodes, of which   21 are leaves
- depth  11 has   42 nodes, of which   23 are leaves
- depth  12 has   38 nodes, of which   17 are leaves
- depth  13 has   42 nodes, of which   21 are leaves
- depth  14 has   42 nodes, of which   25 are leaves
- depth  15 has   34 nodes, of which   18 are leaves
- depth  16 has   32 nodes, of which   14 are leaves
- depth  17 has   36 nodes, of which   23 are leaves
- dep

# Part 3: Random forest

## 3.1: Train a random forest with default settings

In [24]:
from sklearn.ensemble import RandomForestClassifier

In [25]:
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101)

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [26]:

# Fit the forest on training data
simple_forest.fit(X_train, y_train)

print("Simple Random Forest trained successfully.")
print(f"Training accuracy: {simple_forest.score(X_train, y_train):.3f}")
print(f"Validation accuracy: {simple_forest.score(X_valid, y_valid):.3f}")


Simple Random Forest trained successfully.
Training accuracy: 0.819
Validation accuracy: 0.797


## Table: Feature Importances

### Table
**Sample Output** (Feel free to print all words and organize them in any software)

|**Important Words**|**Unimportant Words**|
|:-:|:-:|
|I1 |  U1  |
|I2 |  U2  |
|I3 |  U3  |
|I4 |  U4  |


In [27]:
# --- Feature Importance Analysis (Side-by-Side Table) ---
importances = simple_forest.feature_importances_
feature_names = np.array(vocab_list)

# Create importance DataFrame
imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False).reset_index(drop=True)

# Top 10 most important words
top10 = imp_df.head(10).reset_index(drop=True).rename(
    columns={"feature": "Top Important Words", "importance": "Importance"}
)

# Features with near-zero importance (< 1e-5)
near_zero_mask = imp_df["importance"] < 1e-5
near_zero_features = imp_df[near_zero_mask]
num_near_zero = len(near_zero_features)

# Randomly sample 10 near-zero importance terms
np.random.seed(101)
sample_near_zero = near_zero_features.sample(n=min(10, num_near_zero), random_state=101)
sample_near_zero = sample_near_zero.reset_index(drop=True).rename(
    columns={"feature": "Near-Zero Importance Words", "importance": "Importance"}
)

# Combine both tables side-by-side
side_by_side = pd.concat([top10, sample_near_zero], axis=1)

print(f"\nNumber of features with importance < 1e-5: {num_near_zero}\n")
print("Top 10 Most Important vs. 10 Near-Zero Importance Words:")
display(side_by_side)



Number of features with importance < 1e-5: 7296

Top 10 Most Important vs. 10 Near-Zero Importance Words:


,Top Important Words,Importance,Near-Zero Importance Words,Importance
0,return,0.032990,the_few,0.0
1,excel,0.029485,librari,0.0
2,great,0.028984,an_easy,0.0
3,worst,0.028407,in_my,0.0
4,poor,0.026748,movie_and,0.0
5,disappoint,0.024952,i_even,0.0
6,your_money,0.018002,pet,0.0
7,i_love,0.017955,written,0.0
8,the_best,0.017662,so_when,0.0
9,don't,0.017654,i,0.0




## 3.2 Hyperparameter Tuning for Random Forest



In [29]:
from sklearn.model_selection import GridSearchCV


In [30]:


param_grid_rf = {
    "max_features": [3, 10, 33, 100, 333],
    "max_depth": [16, 32],
    "min_samples_leaf": [1],
    "n_estimators": [100],
    "random_state": [101]
}

rf_search = GridSearchCV(
    estimator=RandomForestClassifier(criterion='gini'),
    param_grid=param_grid_rf,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    n_jobs=-1
)

### Do the search!

In [31]:
print("\nRunning Random Forest Grid Search ...")

rf_search.fit(X_all, y_all)


Running Random Forest Grid Search ...


,estimator,RandomForestClassifier()
,param_grid,"{'max_depth': [16, 32], 'max_features': [3, 10, ...], 'min_samples_leaf': [1], 'n_estimators': [100], ...}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,False
,cv,"PredefinedSpl...hape=(7138,)))"
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,n_estimators,100


### Display search results

In [32]:

print("Random Forest Grid Search Complete.")
print(f"Best balanced accuracy: {rf_search.best_score_:.4f}")
print("Best parameters:", rf_search.best_params_)


Random Forest Grid Search Complete.
Best balanced accuracy: 0.8512
Best parameters: {'max_depth': 32, 'max_features': 33, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}


- Best parameters: {'max_depth': 32, 'max_features': 33, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}

- The value of max_features of best forest is 33

In [ ]:
print("Max possible value for max_features for this dataset is " , X_train.shape[1])

Max possible value for max_features for this dataset  7729


####  What is the maximum possible value for max_features for this dataset? Why is it beneficial to tune this hyperparameter?
The maximum possible value for max_features is the total number of features (i.e., the number of columns in your input data, X_train.shape[1])- 7729.

Tuning max_features is beneficial because:

- A smaller value increases randomness and diversity among trees, which can reduce overfitting and improve generalization.
- A larger value allows each tree to consider more features, which can increase accuracy but may lead to more correlated trees and potential overfitting.
- The optimal value balances tree diversity and predictive power.
- Tuning max_features can help control the bias-variance tradeoff: lower values may increase bias but reduce variance, while higher values may decrease bias but increase variance.
- It can improve computational efficiency: using fewer features per split can make each tree faster to build and reduce memory usage.
- Proper tuning can lead to better feature selection, as trees are forced to consider different subsets of features, potentially highlighting more informative or robust predictors.



#### When fitting random forests, what is the primary tradeoff controlled by the n_estimators hyperparameter? Can you overfit by setting it to be too large? Why or why not?

The n_estimators hyperparameter controls the number of trees in the forest.

- The primary tradeoff is between model stability/performance and computational cost:
  - More trees generally improve performance and reduce variance, up to a point.
  - Too few trees can lead to high variance (unstable predictions).
  - Too many trees increase computation time and memory usage.
- In theory, we cannot overfit by setting n_estimators too large, because random forests average the predictions of many trees, and adding more trees only stabilizes the ensemble.
- After a certain point, adding more trees gives diminishing returns in terms of accuracy or variance reduction, but the computational cost (training time and memory usage) continues to increase.
- Increasing n_estimators can help smooth out randomness from bootstrapping and feature selection, leading to more reliable and reproducible results.
- Larger forests can provide more stable feature importance estimates.
- In practice, the main limitation is computational resources, not overfitting.
- For very large datasets, using too many trees can slow down both training and prediction, so it's important to balance accuracy gains with efficiency.

### Build the best random forest using the best hyperparameters found in 2B 

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [33]:

best_rf_params = rf_search.best_params_
# Note: rf_search.best_params_ contains exactly the keys provided in param_grid_rf
best_rf = RandomForestClassifier(
    n_estimators=best_rf_params['n_estimators'],
    max_features=best_rf_params['max_features'],
    max_depth=best_rf_params['max_depth'],
    min_samples_leaf=best_rf_params['min_samples_leaf'],
    random_state=best_rf_params.get('random_state', 101),
    criterion='gini',
    min_samples_split=2,
    bootstrap=True,
    n_jobs=-1
)
best_rf.fit(X_train, y_train)

print("Best Random Forest trained.")
print(f"Training accuracy: {best_rf.score(X_train, y_train):.3f}")
print(f"Validation accuracy: {best_rf.score(X_valid, y_valid):.3f}")


Best Random Forest trained.
Training accuracy: 0.964
Validation accuracy: 0.851


### Table 3: Comparison of methods on the bag-of-words to sentiment classification task.

Please report **balanced accuracy** on the train, valid, and test sets, to 3 digits of precision

**Sample Output** (Feel free to print all values and organize them by hand)

|**method**|**max depth**|**num trees**|**train BAcc**|**valid BAcc**|**test BAcc**|
|:-|:-:|:-:|:-:|:-:|:-:|
|simple Tree	| - | - | -	|-	|-|
|best Tree	| - | - | -	|-	|-|
|simple RandomForest	| - | - | -	|-	|-|
|best RandomForest	| - | - | -	|-	|-|

In [34]:
from sklearn.metrics import balanced_accuracy_score

models = {
    "Simple DT": base_tree,   # still the original trained max_depth=3 tree
    "Best DT": best_tree,     # newly-trained best dt (separate object)
    "Simple RF": simple_forest, # still the original trained RF with max_depth=3
    "Best RF": best_rf
}

rows = []
for name, model in models.items():
    y_train_pred = model.predict(X_train)
    y_valid_pred = model.predict(X_valid)
    y_test_pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "max_depth": getattr(model, "max_depth", None),
        "n_estimators / num trees": getattr(model, "n_estimators", 1),
        "Train Balanced Acc": balanced_accuracy_score(y_train, y_train_pred),
        "Valid Balanced Acc": balanced_accuracy_score(y_valid, y_valid_pred),
        "Test Balanced Acc": balanced_accuracy_score(y_test, y_test_pred)
    })

summary_df = pd.DataFrame(rows)
display(summary_df)


,Model,max_depth,n_estimators / num trees,Train Balanced Acc,Valid Balanced Acc,Test Balanced Acc
0,Simple DT,3,1,0.645773,0.644560,0.645849
1,Best DT,32,1,0.877218,0.732380,0.748899
2,Simple RF,3,100,0.819126,0.797170,0.778218
3,Best RF,32,100,0.964402,0.851204,0.836933


### Summary and Conclusions

- **Random Forest models (RF)** consistently outperform Decision Trees (DT) across all datasets, achieving higher balanced accuracy on train, validation, and test sets.
- **Best RF** (with tuned hyperparameters) achieves the highest performance overall, with a test balanced accuracy of **0.837**, indicating strong generalization and robustness.
- **Best DT** (tuned) shows significant improvement over the Simple DT, but still lags behind both RF models, especially on the validation and test sets.
- **Simple RF** (default settings) already surpasses both DT models, demonstrating the effectiveness of ensembling even without hyperparameter tuning.
- **Best RF** shows some overfitting (very high train accuracy), but maintains a strong margin on validation and test sets, suggesting the model generalizes well.
- **Increasing max_depth** and using more trees (n_estimators) both contribute to improved performance, but the largest gains come from using the Random Forest ensemble.
- **Hyperparameter tuning** (such as max_depth, min_samples_leaf, and max_features) is crucial for maximizing model performance and preventing overfitting or underfitting.
- **Random Forests** are less sensitive to noise and outliers compared to single trees, making them more reliable for real-world data.
- **Feature importance analysis** from Random Forests can provide valuable insights into which words or features are most predictive for sentiment classification.


**Conclusion:**  
Tuning hyperparameters and using ensemble methods like Random Forests substantially improve model performance for sentiment classification on bag-of-words features. The Best Random Forest model is the most accurate, robust, and reliable choice among those tested.